In [0]:
%run /Users/rahulpatel@cyntexa.com/IAP_Practice/freshkart/00_set_up

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ── Schemas for JSON string columns in bronze.orders ──
delivery_schema = "struct<promised_mins:int, actual_mins:int, delivery_fee:double>"
payment_schema  = "struct<method:string, txn_ref:string>"
items_schema    = "array<struct<product_id:string, qty:string, unit_price:double, discount:double>>"

# ── Read bronze orders and parse JSON string columns ──
df_bronze = spark.table(f"{catalog}.bronze.orders")

df_parsed = (df_bronze
    .withColumn("delivery_parsed", from_json(col("delivery"), delivery_schema))
    .withColumn("payment_parsed",  from_json(col("payment"),  payment_schema))
    .withColumn("items_parsed",    from_json(col("items"),     items_schema))
)

# ── Apply RSU (trim + uppercase) to required string fields ──
df_rsu = (df_parsed
    .withColumn("order_id_clean",     upper(trim(col("order_id"))))
    .withColumn("customer_id_clean",   upper(trim(col("customer_id"))))
    .withColumn("store_id_clean",      upper(trim(col("store_id"))))
    .withColumn("order_status_clean",  upper(trim(col("order_status"))))
    .withColumn("payment_method",      upper(trim(col("payment_parsed.method"))))
    .withColumn("channel",             lit("UNKNOWN"))  # No channel in source → default UNKNOWN
)

# ── Parse timestamps (order_ts accepts space or 'T' separator) ──
df_ts = (df_rsu
    .withColumn("order_ts_ts",
        expr("coalesce(try_to_timestamp(order_ts, 'yyyy-MM-dd HH:mm:ss'), try_to_timestamp(order_ts, \"yyyy-MM-dd'T'HH:mm:ss\"))"))
    .withColumn("updated_at_ts",
        expr("try_to_timestamp(updated_at, 'yyyy-MM-dd HH:mm:ss')"))
)

# ── Apply RSU to item product_ids and default discount to 0 ──
df_items = (df_ts
    .withColumn("items_clean",
        expr("transform(items_parsed, x -> named_struct('product_id', upper(trim(x.product_id)), 'qty', x.qty, 'unit_price', x.unit_price, 'discount', coalesce(x.discount, 0.0)))"))
)

# ── Build reject_reason (concat_ws skips NULLs → valid records get empty string) ──
df_validated = (df_items
    .withColumn(
        "reject_reason",
        concat_ws("; ",
            # Required field checks
            when(col("order_id_clean").isNull() | (col("order_id_clean") == ""), lit("Missing order_id")),
            when(col("customer_id_clean").isNull() | (col("customer_id_clean") == ""), lit("Missing customer_id")),
            when(col("store_id_clean").isNull() | (col("store_id_clean") == ""), lit("Missing store_id")),
            when(col("order_status_clean").isNull(), lit("Missing order_status")),
            when((col("order_status_clean").isNotNull()) & (~col("order_status_clean").isin("PLACED", "DELIVERED", "CANCELLED")),
                 concat(lit("Invalid order_status: "), col("order_status_clean"))),
            when(col("payment_method").isNull() | (col("payment_method") == ""), lit("Missing payment_method")),
            # Timestamp validations
            when(col("order_ts_ts").isNull(), lit("Invalid order_ts format")),
            when(col("updated_at_ts").isNull(), lit("Invalid updated_at format")),
            # Items validation
            when(col("items_parsed").isNull() | (size(col("items_parsed")) == 0), lit("Missing or empty items")),
            when(expr("exists(items_clean, x -> x.product_id IS NULL OR x.product_id = '')"),
                 lit("Missing product_id in items")),
            when(expr("exists(items_clean, x -> x.qty IS NULL OR try_cast(x.qty AS INT) IS NULL OR try_cast(x.qty AS INT) <= 0)"),
                 lit("Invalid item quantity (must be whole number > 0)")),
            when(expr("exists(items_clean, x -> x.unit_price IS NULL OR x.unit_price <= 0 OR x.unit_price % 1 != 0)"),
                 lit("Invalid unit_price (must be whole number > 0)")),
        )
    )
    # Convert empty string to NULL (concat_ws returns '' when all inputs are NULL)
    .withColumn(
        "reject_reason",
        when(col("reject_reason") == "", lit(None).cast("string")).otherwise(col("reject_reason"))
    )
)

# print(f"Total records:       {df_validated.count()}")
# print(f"Valid records:       {df_validated.filter(col('reject_reason').isNull()).count()}")
# print(f"Quarantined records:  {df_validated.filter(col('reject_reason').isNotNull()).count()}")
# display(df_validated.filter(col("reject_reason").isNotNull()).select("order_id_clean", "reject_reason").limit(20))

In [0]:
# ── Split into valid and quarantined ──
df_valid = df_validated.filter(col("reject_reason").isNull())
df_quarantine = df_validated.filter(col("reject_reason").isNotNull())

# ── Deduplicate by order_id, keeping record with latest updated_at ──
dedup_window = Window.partitionBy("order_id_clean").orderBy(col("updated_at_ts").desc())
df_valid_dedup = (df_valid
    .withColumn("_rn", row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# print(f"Valid records (pre-dedup):  {df_valid.count()}")
# print(f"Valid records (post-dedup): {df_valid_dedup.count()}")

# ── Prepare silver.orders ──
df_silver_orders = (df_valid_dedup.select(
    col("order_id_clean").alias("order_id"),
    col("customer_id_clean").alias("customer_id"),
    col("store_id_clean").alias("store_id"),
    col("order_ts_ts").alias("order_ts"),
    to_date(col("order_ts_ts")).alias("order_date"),
    col("order_status_clean").alias("order_status"),
    col("payment_method"),
    col("channel"),
    col("delivery_parsed.promised_mins").alias("promised_mins"),
    col("delivery_parsed.actual_mins").alias("actual_mins"),
    col("delivery_parsed.delivery_fee").cast("decimal(10,2)").alias("delivery_fee"),
    col("updated_at_ts").alias("updated_at"),
))

# ── Prepare silver.order_items (explode with 1-based line_number) ──
df_silver_order_items = (df_valid_dedup
    .select(col("order_id_clean").alias("order_id"), posexplode(col("items_clean")))
    .select(
        col("order_id"),
        (col("pos") + 1).alias("line_number"),
        col("col.product_id").alias("product_id"),
        col("col.qty").cast("int").alias("quantity"),
        col("col.unit_price").cast("decimal(10,2)").alias("unit_price"),
        col("col.discount").cast("decimal(10,2)").alias("discount_amount"),
    )
    .withColumn("gross_amount", (col("quantity") * col("unit_price")).cast("decimal(10,2)"))
    .withColumn("net_amount", (col("gross_amount") - col("discount_amount")).cast("decimal(10,2)"))
)

# ── Prepare quarantine records (raw_record = full original bronze row as JSON) ──
bronze_cols = df_bronze.columns
df_quarantine_prepared = (df_quarantine.select(
    col("order_id_clean").alias("order_id"),
    col("reject_reason"),
    col("updated_at_ts").alias("updated_at"),
    to_json(struct(*[col(c) for c in bronze_cols])).alias("raw_record"),
    current_timestamp().alias("quarantine_at"),
))

# print(f"\nSilver orders:       {df_silver_orders.count()}")
# print(f"Silver order_items:  {df_silver_order_items.count()}")
# print(f"Quarantine records:  {df_quarantine_prepared.count()}")

# display(df_silver_orders.limit(5))
# display(df_silver_order_items.limit(10))

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.orders (
    order_id        STRING,
    customer_id     STRING,
    store_id        STRING,
    order_ts        TIMESTAMP,
    order_date      DATE,
    order_status    STRING,
    payment_method  STRING,
    channel         STRING,
    promised_mins   INT,
    actual_mins     INT,
    delivery_fee    DECIMAL(10,2),
    updated_at      TIMESTAMP
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.order_items (
    order_id        STRING,
    line_number     INT,
    product_id      STRING,
    quantity        INT,
    unit_price      DECIMAL(10,2),
    discount_amount DECIMAL(10,2),
    gross_amount    DECIMAL(10,2),
    net_amount      DECIMAL(10,2)
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.quarantine_orders (
    order_id        STRING,
    reject_reason   STRING,
    updated_at      TIMESTAMP,
    raw_record      STRING,
    quarantine_at   TIMESTAMP
)
""")


In [0]:
# ── Register temp views for MERGE ──
df_silver_orders.createOrReplaceTempView("vw_silver_orders")
df_silver_order_items.createOrReplaceTempView("vw_silver_order_items")
df_quarantine_prepared.createOrReplaceTempView("vw_quarantine")

# ── MERGE 1: silver.orders (match on order_id, update only when updated_at is newer) ──
spark.sql(f"""
MERGE INTO {catalog}.silver.orders AS t
USING vw_silver_orders AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
# print("✓ silver.orders merged")

# ── MERGE 2a: Delete all items for orders being re-processed (removes obsolete line items) ──
spark.sql(f"""
MERGE INTO {catalog}.silver.order_items AS t
USING (SELECT DISTINCT order_id FROM vw_silver_order_items) AS s
ON t.order_id = s.order_id
WHEN MATCHED THEN DELETE
""")
# print("✓ silver.order_items obsolete items deleted")

# ── MERGE 2b: Upsert order_items (match on order_id + line_number) ──
spark.sql(f"""
MERGE INTO {catalog}.silver.order_items AS t
USING vw_silver_order_items AS s
ON t.order_id = s.order_id AND t.line_number = s.line_number
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
# print("✓ silver.order_items upserted")

# ── MERGE 3: quarantine_orders (idempotent via null-safe match on order_id + raw_record) ──
spark.sql(f"""
MERGE INTO {catalog}.silver.quarantine_orders AS t
USING vw_quarantine AS s
ON t.order_id <=> s.order_id AND t.raw_record <=> s.raw_record
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
# print("✓ silver.quarantine_orders merged")

In [0]:
# print("═══ Silver Layer Summary ═══")
# print(f"silver.orders:           {spark.table(f'{catalog}.silver.orders').count()} rows")
# print(f"silver.order_items:       {spark.table(f'{catalog}.silver.order_items').count()} rows")
# print(f"silver.quarantine_orders:  {spark.table(f'{catalog}.silver.quarantine_orders').count()} rows")

# print("\n── Sample silver.orders ──")
# display(spark.table(f"{catalog}.silver.orders").orderBy("order_id").limit(5))

# print("\n── Sample silver.order_items ──")
# display(spark.table(f"{catalog}.silver.order_items").orderBy("order_id", "line_number").limit(10))

# print("\n── Quarantined records ──")
# display(spark.table(f"{catalog}.silver.quarantine_orders").select("order_id", "reject_reason").limit(20))

In [0]:
from pyspark.sql.functions import *

bronze_stores = spark.table(f"{catalog}.bronze.stores")

silver_stores = bronze_stores.select(
    upper(trim(col("store_id"))).alias("store_id"),       # string, RSU
    trim(col("store_name")).alias("store_name"),           # string, RS
    trim(col("city")).alias("city"),                       # string, RS
    trim(col("state")).alias("state"),                     # string, RS
    trim(col("region")).alias("region"),                   # string, RS
    to_date(col("opening_date")).alias("opening_date"),    # date
)

(
    silver_stores.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.stores")
)


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --- 1. Create silver table with Change Data Feed enabled ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.silver.products (
        product_id STRING,
        product_name STRING,
        brand STRING,
        category_l1 STRING,
        category_l2 STRING,
        packsize INT,
        pack_unit STRING,
        mrp DECIMAL(10,2),
        selling_price DECIMAL(10,2),
        is_active BOOLEAN,
        tags ARRAY<STRING>,
        last_changed_snapshot_date DATE
    )
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --- 1. Create silver table with SCD Type 2 columns ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.silver.customers (
        customer_id STRING,
        full_name STRING,
        email STRING,
        phone STRING,
        city STRING,
        tier STRING,
        signup_date DATE,
        effective_from DATE,
        effective_to DATE,
        is_current BOOLEAN
    )
""")

# --- 2. Read & transform bronze customers ---
bronze_customers = spark.table(f"{catalog}.bronze.customers")

silver_customers = bronze_customers.select(
    upper(trim(col("customer_id"))).alias("customer_id"),                              # string, RSU
    trim(col("full_name")).alias("full_name"),                                   # string, RS
    lower(trim(col("email"))).alias("email"),                                    # string, RS + lower
    substring(regexp_replace(col("phone"), "[^0-9]", ""), -10, 10).alias("phone"),  # last 10 digits
    trim(col("city")).alias("city"),                                             # string, RS
    upper(trim(col("tier"))).alias("tier"),                                      # string, RSU
    coalesce(
        try_to_date(col("signup_date"), "yyyy-MM-dd"),
        try_to_date(col("signup_date"), "dd/MM/yyyy")
    ).alias("signup_date"),                                                      # date, both formats
    to_date(regexp_extract(col("_source_file"), r"(\d{4}-\d{2}-\d{2})", 1)).alias("snapshot_date"),
)

# --- 3. Dedup by customer_id, keep latest snapshot ---
window_spec = Window.partitionBy("customer_id").orderBy(col("snapshot_date").desc())
silver_customers_dedup = (
    silver_customers
    .withColumn("_rn", row_number().over(window_spec))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# --- 4. Determine effective_from ---
# New customers (not in silver): effective_from = signup_date (first date this version applies)
# Existing customers with tracked changes: effective_from = snapshot_date (date new version applies)
existing_ids = spark.sql(
    f"SELECT DISTINCT customer_id, True AS _exists FROM {catalog}.silver.customers WHERE is_current = true"
)

source_flagged = silver_customers_dedup.join(existing_ids, "customer_id", "left")

source_final = source_flagged.withColumn(
    "effective_from",
    when(col("_exists").isNull(), col("signup_date"))
    .otherwise(col("snapshot_date"))
).drop("_exists")

# --- 5. Identify changed customers (tracked columns only: full_name, email, phone) ---
current = spark.sql(
    f"SELECT customer_id, full_name, email, phone FROM {catalog}.silver.customers WHERE is_current = true"
)

changed_ids = (
    source_final.alias("s")
    .join(current.alias("t"), col("s.customer_id") == col("t.customer_id"), "inner")
    .filter(
        "s.full_name IS DISTINCT FROM t.full_name "
        "OR s.email IS DISTINCT FROM t.email "
        "OR s.phone IS DISTINCT FROM t.phone"
    )
    .select(col("s.customer_id").alias("customer_id"))
    .distinct()
)

# New version rows for changed customers (will be inserted after MERGE closes old rows)
new_versions = (
    source_final.join(changed_ids, "customer_id", "inner")
    .select(
        "customer_id", "full_name", "email", "phone", "city", "tier", "signup_date",
        "effective_from",
        lit(None).cast("date").alias("effective_to"),
        lit(True).alias("is_current"),
    )
)

# --- 6. MERGE: close old rows (tracked changes) + update non-tracked in place + insert new customers ---
source_for_merge = source_final.select(
    "customer_id", "full_name", "email", "phone", "city", "tier", "signup_date",
    "effective_from",
    lit(None).cast("date").alias("effective_to"),
    lit(True).alias("is_current"),
)

delta_table = DeltaTable.forName(spark, f"{catalog}.silver.customers")

(
    delta_table.alias("t")
    .merge(source_for_merge.alias("s"), "t.customer_id = s.customer_id AND t.is_current = true")
    .whenMatchedUpdate(
        condition=(
            "t.full_name IS DISTINCT FROM s.full_name "
            "OR t.email IS DISTINCT FROM s.email "
            "OR t.phone IS DISTINCT FROM s.phone"
        ),
        set={"effective_to": "s.effective_from", "is_current": "false"}
    )
    .whenMatchedUpdate(
        condition="t.city IS DISTINCT FROM s.city OR t.tier IS DISTINCT FROM s.tier",
        set={"city": "s.city", "tier": "s.tier"}
    )
    .whenNotMatchedInsertAll()
    .execute()
)

# --- 7. INSERT new version rows for changed customers ---
n_new_versions = new_versions.count()
if n_new_versions > 0:
    new_versions.write.mode("append").saveAsTable(f"{catalog}.silver.customers")

# print(f"New version rows inserted: {n_new_versions}")

# # --- 8. Display results ---
# display(spark.sql(f"SELECT * FROM {catalog}.silver.customers ORDER BY customer_id, effective_from"))
# --- 2. Read & transform bronze products ---
bronze_products = spark.table(f"{catalog}.bronze.products")

silver_products = bronze_products.select(
    upper(trim(col("product_id"))).alias("product_id"),                      # string, RSU
    trim(col("product_name")).alias("product_name"),                       # string
    trim(col("brand")).alias("brand"),                                      # string
    trim(get_json_object(col("category"), "$.l1")).alias("category_l1"),    # string
    trim(get_json_object(col("category"), "$.l2")).alias("category_l2"),    # string
    get_json_object(col("pack"), "$.size").cast("int").alias("packsize"),  # int
    trim(get_json_object(col("pack"), "$.unit")).alias("pack_unit"),        # string
    col("mrp").cast("decimal(10,2)").alias("mrp"),                         # decimal
    col("selling_price").cast("decimal(10,2)").alias("selling_price"),     # decimal
    col("is_active").cast("boolean").alias("is_active"),                   # boolean
    from_json(col("tags"), "array<string>").alias("tags"),                 # array<string>
    to_date(col("snapshot_date")).alias("last_changed_snapshot_date"),      # date
)

# --- 3. Dedup by product_id, keep latest snapshot ---
window_spec = Window.partitionBy("product_id").orderBy(col("last_changed_snapshot_date").desc())
silver_products_dedup = (
    silver_products
    .withColumn("_rn", row_number().over(window_spec))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# --- 4. MERGE INTO silver (SCD Type 1) ---
# Update only when at least one value changed (excluding last_changed_snapshot_date)
delta_table = DeltaTable.forName(spark, f"{catalog}.silver.products")

update_cols = [c for c in silver_products_dedup.columns if c != "product_id"]
update_set = {c: f"s.{c}" for c in update_cols}

merge_condition = " OR ".join([
    f"t.{c} IS DISTINCT FROM s.{c}"
    for c in update_cols
    if c != "last_changed_snapshot_date"
])

(
    delta_table.alias("t")
    .merge(silver_products_dedup.alias("s"), "t.product_id = s.product_id")
    .whenMatchedUpdate(condition=merge_condition, set=update_set)
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --- 1. Create silver table with SCD Type 2 columns ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.silver.customers (
        customer_id STRING,
        full_name STRING,
        email STRING,
        phone STRING,
        city STRING,
        tier STRING,
        signup_date DATE,
        effective_from DATE,
        effective_to DATE,
        is_current BOOLEAN
    )
""")

# --- 2. Read & transform bronze customers ---
bronze_customers = spark.table(f"{catalog}.bronze.customers")

silver_customers = bronze_customers.select(
    upper(trim(col("customer_id"))).alias("customer_id"),                              # string, RSU
    trim(col("full_name")).alias("full_name"),                                   # string, RS
    lower(trim(col("email"))).alias("email"),                                    # string, RS + lower
    substring(regexp_replace(col("phone"), "[^0-9]", ""), -10, 10).alias("phone"),  # last 10 digits
    trim(col("city")).alias("city"),                                             # string, RS
    upper(trim(col("tier"))).alias("tier"),                                      # string, RSU
    coalesce(
        try_to_date(col("signup_date"), "yyyy-MM-dd"),
        try_to_date(col("signup_date"), "dd/MM/yyyy")
    ).alias("signup_date"),                                                      # date, both formats
    to_date(regexp_extract(col("_source_file"), r"(\d{4}-\d{2}-\d{2})", 1)).alias("snapshot_date"),
)

# --- 3. Dedup by customer_id, keep latest snapshot ---
window_spec = Window.partitionBy("customer_id").orderBy(col("snapshot_date").desc())
silver_customers_dedup = (
    silver_customers
    .withColumn("_rn", row_number().over(window_spec))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# --- 4. Determine effective_from ---
# New customers (not in silver): effective_from = signup_date (first date this version applies)
# Existing customers with tracked changes: effective_from = snapshot_date (date new version applies)
existing_ids = spark.sql(
    f"SELECT DISTINCT customer_id, True AS _exists FROM {catalog}.silver.customers WHERE is_current = true"
)

source_flagged = silver_customers_dedup.join(existing_ids, "customer_id", "left")

source_final = source_flagged.withColumn(
    "effective_from",
    when(col("_exists").isNull(), col("signup_date"))
    .otherwise(col("snapshot_date"))
).drop("_exists")

# --- 5. Identify changed customers (tracked columns only: full_name, email, phone) ---
current = spark.sql(
    f"SELECT customer_id, full_name, email, phone FROM {catalog}.silver.customers WHERE is_current = true"
)

changed_ids = (
    source_final.alias("s")
    .join(current.alias("t"), col("s.customer_id") == col("t.customer_id"), "inner")
    .filter(
        "s.full_name IS DISTINCT FROM t.full_name "
        "OR s.email IS DISTINCT FROM t.email "
        "OR s.phone IS DISTINCT FROM t.phone"
    )
    .select(col("s.customer_id").alias("customer_id"))
    .distinct()
)

# New version rows for changed customers (will be inserted after MERGE closes old rows)
new_versions = (
    source_final.join(changed_ids, "customer_id", "inner")
    .select(
        "customer_id", "full_name", "email", "phone", "city", "tier", "signup_date",
        "effective_from",
        lit(None).cast("date").alias("effective_to"),
        lit(True).alias("is_current"),
    )
)

# --- 6. MERGE: close old rows (tracked changes) + update non-tracked in place + insert new customers ---
source_for_merge = source_final.select(
    "customer_id", "full_name", "email", "phone", "city", "tier", "signup_date",
    "effective_from",
    lit(None).cast("date").alias("effective_to"),
    lit(True).alias("is_current"),
)

delta_table = DeltaTable.forName(spark, f"{catalog}.silver.customers")

(
    delta_table.alias("t")
    .merge(source_for_merge.alias("s"), "t.customer_id = s.customer_id AND t.is_current = true")
    .whenMatchedUpdate(
        condition=(
            "t.full_name IS DISTINCT FROM s.full_name "
            "OR t.email IS DISTINCT FROM s.email "
            "OR t.phone IS DISTINCT FROM s.phone"
        ),
        set={"effective_to": "s.effective_from", "is_current": "false"}
    )
    .whenMatchedUpdate(
        condition="t.city IS DISTINCT FROM s.city OR t.tier IS DISTINCT FROM s.tier",
        set={"city": "s.city", "tier": "s.tier"}
    )
    .whenNotMatchedInsertAll()
    .execute()
)

# --- 7. INSERT new version rows for changed customers ---
n_new_versions = new_versions.count()
if n_new_versions > 0:
    new_versions.write.mode("append").saveAsTable(f"{catalog}.silver.customers")

# print(f"New version rows inserted: {n_new_versions}")

# # --- 8. Display results ---
# display(spark.sql(f"SELECT * FROM {catalog}.silver.customers ORDER BY customer_id, effective_from"))